<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

<a href="https://colab.research.google.com/github/yhilpisch/algocolab/blob/main/notebooks/03_cloud_deployment_monitoring.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Algorithmic Trading with Python & Google Colab
## Session 3: From Notebook to Trading System: Cloud Deployment, Monitoring & Live Simulation

&copy; Dr. Yves J. Hilpisch<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com

---

### Objectives
1. **Architectural Separation**: Loading model weights and scaler onto CPU.
2. **Canonical Feature Pipeline**: Standardizing live features to match training.
3. **Real-Time ZeroMQ Streaming**: Publishing and consuming market push events.
4. **Transactional SQLite Persistence**: Decoupled database recording.
5. **Live Trading Engine & Circuit Breakers**: Real-time inferences & risk halts.
6. **Real-Time Monitoring Dashboard**: Live visual telemetry of NAV & drawdown.


In [ ]:
import time
import threading
import sqlite3
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zmq
import torch
import torch.nn as nn
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

style_name = (
    'seaborn-v0_8'
    if 'seaborn-v0_8' in plt.style.available
    else 'default'
)
plt.style.use(style_name)
plt.rcParams['figure.figsize'] = (12, 6)
print("Session 3 Deployment Environment Initialized.")


## 1. Production Model Architecture & Scaler Loading

Production inference runs on standard CPU. We load the model weights and
the calibrated feature scaler (`scaler_mean`, `scaler_scale`).


In [ ]:
class ProductionDNN(nn.Module):
    def __init__(self, input_dim=7, hidden_units=[64, 32]):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_units:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


prod_model = ProductionDNN(input_dim=7, hidden_units=[64, 32])
scaler_mean = None
scaler_scale = None

if Path('best_trading_dnn.pt').exists():
    try:
        checkpoint = torch.load(
            'best_trading_dnn.pt', map_location='cpu', weights_only=False
        )
    except TypeError:
        checkpoint = torch.load('best_trading_dnn.pt', map_location='cpu')

    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        prod_model.load_state_dict(checkpoint["state_dict"])
        scaler_mean = checkpoint.get("scaler_mean")
        scaler_scale = checkpoint.get("scaler_scale")
        if isinstance(scaler_mean, torch.Tensor):
            scaler_mean = scaler_mean.numpy()
        if isinstance(scaler_scale, torch.Tensor):
            scaler_scale = scaler_scale.numpy()
        print("Loaded model artifact and scaler from 'best_trading_dnn.pt'.")
    elif isinstance(checkpoint, dict):
        prod_model.load_state_dict(checkpoint)
        print("Loaded model state dict.")
else:
    print("Running with initialized weights and unit scaler for demonstration.")

prod_model.eval()


## 2. Canonical Feature Extractor (Shared with Training)

To prevent feature ordering or calculation mismatch between training and live
inference, we use the canonical feature extraction helper:
$$X_t = [r_{t-1}, r_{t-2}, r_{t-3}, r_{t-4}, r_{t-5}, \sigma_{t-1}^{(20)}, \mu_{t-1}^{(10)}]$$


In [ ]:
def build_feature_vector(
    prices_window, lags=5, vol_window=20, mom_window=10
):
    p_arr = np.asarray(prices_window, dtype=np.float64)
    log_rets = np.diff(np.log(p_arr))

    lag_feats = [float(log_rets[-i]) for i in range(1, lags + 1)]
    vol_feat = float(np.std(log_rets[-vol_window:], ddof=1))
    mom_feat = float(np.mean(log_rets[-mom_window:]))

    return np.array(lag_feats + [vol_feat, mom_feat], dtype=np.float32)


print("Canonical feature extractor defined.")


## 3. Real-Time ZeroMQ Streaming Server

We define the ZeroMQ market publisher broadcasting JSON ticks over
`tcp://127.0.0.1:5555`.


In [ ]:
def run_tick_server(
    bind_addr="tcp://127.0.0.1:5555",
    symbol="EURUSD",
    start_price=1.1000,
    dt=0.01,
    sigma=0.0002,
    max_ticks=200
):
    ctx = zmq.Context()
    socket = ctx.socket(zmq.PUB)
    socket.setsockopt(zmq.LINGER, 0)
    try:
        socket.bind(bind_addr)
    except zmq.ZMQError:
        socket.close(0)
        ctx.term()
        return

    price = start_price
    rng = np.random.default_rng(42)
    for _ in range(max_ticks):
        shock = rng.normal(0.0, sigma * price)
        price = max(1.0, price + shock)
        payload = {
            "time": datetime.now(timezone.utc).isoformat(),
            "symbol": symbol,
            "price": round(float(price), 4)
        }
        try:
            socket.send_json(payload)
        except (zmq.ContextTerminated, zmq.ZMQError):
            break
        time.sleep(dt)
    socket.close(0)
    ctx.term()


print("ZeroMQ Tick Server function ready.")


## 4. Real-Time Trading Client with Scaling & Circuit Breakers

The `ZMQTradingClient` standardizes streaming features, marks NAV to market,
and trips automated circuit breakers if drawdown exceeds the risk ceiling.


In [ ]:
class ZMQTradingClient:
    def __init__(
        self,
        connect_addr="tcp://127.0.0.1:5555",
        model=prod_model,
        scaler_mean=scaler_mean,
        scaler_scale=scaler_scale,
        initial_capital=100_000.0,
        max_drawdown_limit=0.10,
        tc_rate=0.0005,
        db_path="webinar_live_trading.db"
    ):
        self.connect_addr = connect_addr
        self.model = model.eval()
        self.scaler_mean = scaler_mean
        self.scaler_scale = scaler_scale
        self.cash = initial_capital
        self.nav = initial_capital
        self.peak_nav = initial_capital
        self.position = 0
        self.units_per_trade = 100
        self.max_dd_limit = max_drawdown_limit
        self.tc_rate = tc_rate
        self.halted = False
        self.prices = []
        self.timestamps = []
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        conn = sqlite3.connect(self.db_path, check_same_thread=False)
        with conn:
            conn.execute(
                "CREATE TABLE IF NOT EXISTS ticks ("
                "id INTEGER PRIMARY KEY AUTOINCREMENT, "
                "timestamp TEXT, symbol TEXT, price REAL)"
            )
            conn.execute(
                "CREATE TABLE IF NOT EXISTS signals ("
                "id INTEGER PRIMARY KEY AUTOINCREMENT, "
                "timestamp TEXT, symbol TEXT, prob_up REAL, signal INTEGER)"
            )
            conn.execute(
                "CREATE TABLE IF NOT EXISTS orders ("
                "id INTEGER PRIMARY KEY AUTOINCREMENT, "
                "timestamp TEXT, symbol TEXT, side TEXT, "
                "units INTEGER, price REAL, cost REAL)"
            )
            conn.execute(
                "CREATE TABLE IF NOT EXISTS portfolio_state ("
                "id INTEGER PRIMARY KEY AUTOINCREMENT, "
                "timestamp TEXT, position INTEGER, cash REAL, "
                "nav REAL, drawdown REAL)"
            )
        conn.close()

    def process_tick(self, timestamp, symbol, price, conn):
        self.prices.append(price)
        self.timestamps.append(timestamp)
        with conn:
            conn.execute(
                "INSERT INTO ticks (timestamp, symbol, price) VALUES (?,?,?)",
                (timestamp, symbol, price)
            )

        # Mark NAV to market
        self.nav = self.cash + (self.position * self.units_per_trade * price)
        if self.nav > self.peak_nav:
            self.peak_nav = self.nav

        if len(self.prices) < 22 or self.halted:
            return

        # Canonical feature extraction (C2)
        feat_vector = build_feature_vector(
            self.prices[-25:], lags=5, vol_window=20, mom_window=10
        )

        # Real-time feature standardization (C3)
        if self.scaler_mean is not None and self.scaler_scale is not None:
            feat_vector = (feat_vector - self.scaler_mean) / self.scaler_scale

        x_t = torch.tensor(feat_vector, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            prob = torch.sigmoid(self.model(x_t)).item()

        # Decision threshold (0.505 / 0.495 for demonstration)
        target_pos = 1 if prob > 0.505 else (-1 if prob < 0.495 else 0)
        with conn:
            conn.execute(
                "INSERT INTO signals "
                "(timestamp, symbol, prob_up, signal) VALUES (?, ?, ?, ?)",
                (timestamp, symbol, prob, target_pos)
            )

        # Order rebalancing
        delta = target_pos - self.position
        if delta != 0:
            units = abs(delta) * self.units_per_trade
            side = "BUY" if delta > 0 else "SELL"
            cost = units * price * self.tc_rate
            self.cash -= (delta * self.units_per_trade * price + cost)
            self.position = target_pos
            with conn:
                conn.execute(
                    "INSERT INTO orders "
                    "(timestamp, symbol, side, units, price, cost) "
                    "VALUES (?, ?, ?, ?, ?, ?)",
                    (timestamp, symbol, side, units, price, cost)
                )

        # Drawdown and circuit breaker check (H1)
        self.nav = self.cash + (self.position * self.units_per_trade * price)
        if self.nav > self.peak_nav:
            self.peak_nav = self.nav
        peak = self.peak_nav
        dd = (peak - self.nav) / peak if peak > 0 else 0.0

        if dd >= self.max_dd_limit:
            self.halted = True
            print(f"[{timestamp}] CIRCUIT BREAKER TRIPPED! Drawdown: {dd:.2%}")

        with conn:
            conn.execute(
                "INSERT INTO portfolio_state "
                "(timestamp, position, cash, nav, drawdown) "
                "VALUES (?, ?, ?, ?, ?)",
                (timestamp, self.position, self.cash, self.nav, dd)
            )

    def run(self, max_ticks=100):
        ctx = zmq.Context()
        socket = ctx.socket(zmq.SUB)
        socket.setsockopt(zmq.LINGER, 0)
        socket.setsockopt(zmq.RCVTIMEO, 500)
        socket.connect(self.connect_addr)
        socket.setsockopt_string(zmq.SUBSCRIBE, "")
        conn = sqlite3.connect(self.db_path, check_same_thread=False)

        count = 0
        idle_timeouts = 0
        while count < max_ticks:
            try:
                tick = socket.recv_json()
                idle_timeouts = 0
            except zmq.Again:
                idle_timeouts += 1
                if idle_timeouts >= 4:
                    break
                continue
            except (zmq.ContextTerminated, zmq.ZMQError):
                break
            self.process_tick(
                tick["time"], tick["symbol"], float(tick["price"]), conn
            )
            count += 1
            if count % 25 == 0:
                t_str = tick['time'][:19]
                p_val = tick['price']
                print(
                    f"[{t_str}] Ticks: {count:03d} | "
                    f"Price: {p_val:.2f} | "
                    f"Pos: {self.position:+d} | "
                    f"NAV: ${self.nav:,.2f}"
                )

        socket.close(0)
        conn.close()
        ctx.term()


# Target financial asset (swappable: 'EURUSD', 'SPY', 'BTC-USD', etc.)
SYMBOL = 'EURUSD'
DATA_URL = "https://hilpisch.com/eod_data.csv"
df_hist = pd.read_csv(DATA_URL, parse_dates=['Date']).set_index('Date')
init_price = round(float(df_hist[SYMBOL].dropna().iloc[-1]), 4)

# Start publisher and consumer together
srv_thread = threading.Thread(
    target=run_tick_server,
    kwargs={
        'bind_addr': 'tcp://127.0.0.1:5555',
        'symbol': SYMBOL,
        'start_price': init_price,
        'dt': 0.01,
        'max_ticks': 150
    },
    daemon=True
)
srv_thread.start()
time.sleep(0.05)

client = ZMQTradingClient(
    connect_addr='tcp://127.0.0.1:5555',
    model=prod_model,
    scaler_mean=scaler_mean,
    scaler_scale=scaler_scale,
    initial_capital=100000.0,
    max_drawdown_limit=0.10,
    db_path="webinar_live_trading.db"
)
client.run(max_ticks=100)


## 5. Live Dashboard & Database Auditing

We query SQLite to inspect executed orders, verify signal probabilities,
and render the operational dashboard with explicit numeric type handling.


In [ ]:
conn = sqlite3.connect("webinar_live_trading.db")

orders_df = pd.read_sql_query("SELECT * FROM orders", conn)
portfolio_df = pd.read_sql_query("SELECT * FROM portfolio_state", conn)
signals_df = pd.read_sql_query("SELECT * FROM signals", conn)
conn.close()

if not orders_df.empty:
    print("--- AUDIT: LAST 5 EXECUTED ORDERS ---")
    print(orders_df.tail())
else:
    print("--- AUDIT: No trade switches occurred in this test window ---")

# Convert SQLite columns explicitly to clean numeric float arrays
nav_series = pd.to_numeric(
    portfolio_df['nav'], errors='coerce'
).fillna(100000.0)
dd_series = pd.to_numeric(
    portfolio_df['drawdown'], errors='coerce'
).fillna(0.0)
x_idx = np.arange(len(portfolio_df))

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=False)

# 1. NAV Growth
axes[0].plot(
    x_idx, nav_series.to_numpy(dtype=float),
    color='#27AE60', linewidth=2, label='Portfolio NAV ($)'
)
axes[0].axhline(
    100000, color='gray', linestyle='--', label='Initial Capital ($100k)'
)
axes[0].set_title('Live Monitoring: Portfolio NAV Evolution')
axes[0].set_ylabel('NAV ($)')
axes[0].legend()

# 2. Drawdown Gauge (passing clean float arrays prevents isfinite TypeError)
axes[1].fill_between(
    x_idx,
    -dd_series.to_numpy(dtype=float) * 100,
    0,
    color='#EB5757',
    alpha=0.4,
    label='Drawdown (%)'
)
axes[1].axhline(
    -10, color='red', linestyle='--', label='Circuit Breaker Limit (-10%)'
)
axes[1].set_title('Live Monitoring: Portfolio Drawdown Gauge')
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend()

# 3. Model Prediction Distribution
if not signals_df.empty:
    sig_probs = pd.to_numeric(signals_df['prob_up'], errors='coerce').dropna()
    axes[2].hist(
        sig_probs.to_numpy(dtype=float),
        bins=30, color='#2F80ED', alpha=0.7, edgecolor='black'
    )
axes[2].axvline(
    0.505, color='green', linestyle='--', label='Long Threshold (0.505)'
)
axes[2].axvline(
    0.495, color='red', linestyle='--', label='Short Threshold (0.495)'
)
axes[2].set_title('Live Monitoring: Model Prediction Distribution')
axes[2].set_xlabel('Predicted Probability P(Up)')
axes[2].legend()

plt.tight_layout()
plt.show()


---
### Full Series Summary
- **Session 1 (Discover)**: Efficient markets, random walks, statistical
  tests, and linear baseline backtesting.
- **Session 2 (Learn)**: PyTorch GPU training, deep neural architectures,
  threshold sensitivity analysis, and checkpointing.
- **Session 3 (Deploy)**: ZeroMQ streaming server/client architecture,
  real-time feature standardization, SQLite persistence, and operational risk.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">